# 03 Documentation audit (D9)

The published schema is our ground truth, so it has to be checked against the files before
it is trusted. A documented column that is absent, or a documented foreign key that does not
resolve, is a finding either way: stale documentation is itself a form of dirtiness.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

In [2]:
from swsn.audit import audit_dataset
from swsn.schema_doc import load_all_schemas
from swsn.tables import load_tables

schemas = load_all_schemas("../ground_truth/schema")
for did, schema in schemas.items():
    tables = load_tables(did, list(schema["tables"]), data_dir="../data")
    report = audit_dataset(did, schema, tables)
    print(f"=== {did}")
    print("   ", report.summary())
    for t in report.tables:
        if t.missing_columns or t.undocumented_columns:
            print(f"    {t.table}: missing={t.missing_columns} undocumented={t.undocumented_columns}")
    for fk in report.foreign_keys:
        print(f"    FK {fk.child_table}.{fk.child_column} -> {fk.parent_table}.{fk.parent_column}"
              f"  {fk.resolving_share:.2%}  nulls={fk.child_nulls}")
    print()

=== instacart
    {'dataset': 'instacart', 'tables': 6, 'mean_column_coverage': 1.0, 'missing_columns': 0, 'undocumented_columns': 0, 'fk_checked': 6, 'fk_satisfied': 6, 'fk_mean_resolving_share': 1.0}
    FK order_products__prior.order_id -> orders.order_id  100.00%  nulls=0
    FK order_products__prior.product_id -> products.product_id  100.00%  nulls=0
    FK order_products__train.order_id -> orders.order_id  100.00%  nulls=0
    FK order_products__train.product_id -> products.product_id  100.00%  nulls=0
    FK products.aisle_id -> aisles.aisle_id  100.00%  nulls=0
    FK products.department_id -> departments.department_id  100.00%  nulls=0



=== olist
    {'dataset': 'olist', 'tables': 9, 'mean_column_coverage': 1.0, 'missing_columns': 0, 'undocumented_columns': 0, 'fk_checked': 6, 'fk_satisfied': 6, 'fk_mean_resolving_share': 1.0}
    FK olist_orders_dataset.customer_id -> olist_customers_dataset.customer_id  100.00%  nulls=0
    FK olist_order_items_dataset.order_id -> olist_orders_dataset.order_id  100.00%  nulls=0
    FK olist_order_items_dataset.product_id -> olist_products_dataset.product_id  100.00%  nulls=0
    FK olist_order_items_dataset.seller_id -> olist_sellers_dataset.seller_id  100.00%  nulls=0
    FK olist_order_payments_dataset.order_id -> olist_orders_dataset.order_id  100.00%  nulls=0
    FK olist_order_reviews_dataset.order_id -> olist_orders_dataset.order_id  100.00%  nulls=0



## Result

Column coverage **1.00** for both datasets, zero missing and zero undocumented columns, and
**all 12 documented foreign keys across the two datasets resolve at 100.00 %**, on tables up
to 32 million child rows.

That is the precondition for the whole C-doc design: if the documentation did not hold, the
ground truth derived from it would be worthless. Row counts are not published, so
`row_count` is null in the schema files and that column of the audit says nothing rather
than guessing a number.

## The BOM

`product_category_name_translation.csv` starts with a UTF-8 byte-order mark
(`ef bb bf`). pandas strips it with the C parser; a plain `csv.DictReader` does not, and the
first column becomes `"\ufeffproduct_category_name"`. `swsn.tables` reads with
`encoding="utf-8-sig"` so the documented name always holds — but anything reading these
files another way will see a phantom column.